# Full Rigetti Campaign on AWS Braket (Cepheus-1-108Q)

The previous device, Rigetti Ankaa-3, has been retired, so this notebook
re-runs the **entire hardware catalog** (Grover, QFT, BV, and BVsb) fresh on
its successor, Cepheus-1-108Q. This gives a complete, directly comparable
dataset on the new device and lets us plot how the hardware has changed
generation to generation.

**This notebook only submits jobs and saves the resulting JSON files** —
it does not rebuild the CSV or regenerate any plots. Once you have the
JSON files you need, run `enrich_dsr_profile.py`, `enrich_hellinger.py`,
`build_csv_from_json.py`, and `differential_success_rate_analysis.py`
separately (from a repo checkout) to fold them into `DSR_result.csv` and
the figures.

Each algorithm has its own **submit** and **retrieve** cell pair below, so
you can run only the ones you want, in any order. Real QPU queues can take
minutes to hours, so submitting never waits for completion — every job is
saved to disk with its AWS job ARN immediately, and the retrieve cells are
safe to re-run as many times as needed while jobs work through the queue.

In [ ]:
# One-time setup: force-installs this exact qward version (with AWS Braket / Rigetti support) from PyPI.
%pip install --force-reinstall --no-cache-dir qiskit-qward[aws]==0.27.0

In [ ]:
import importlib.metadata as md

print(md.version("qiskit-qward"))

In [ ]:
import os
from pathlib import Path

from qward.examples.papers.bv.bv_aws import BVAWSExperiment
from qward.examples.papers.bv.bv_signal_background_aws import BVSignalBackgroundAWSExperiment
from qward.examples.papers.grover.grover_aws import GroverAWSExperiment
from qward.examples.papers.qft.qft_aws import QFTAWSExperiment

## 1. AWS credentials

Don't have an access key yet? See [Managing access keys for IAM users](https://docs.aws.amazon.com/IAM/latest/UserGuide/id_credentials_access-keys.html) for how to create one from the AWS Console (IAM → Users → Security credentials → Create access key). Make sure the IAM user/role has Braket permissions (e.g. `AmazonBraketFullAccess`).

Fill in your Braket-enabled AWS keys below. No shell env vars or `.env`
file needed — this cell sets everything the executor needs for the rest
of the notebook.

In [ ]:
# Fill these in with your own AWS Braket-enabled credentials.
AWS_ACCESS_KEY_ID = "YOUR_AWS_ACCESS_KEY_ID"
AWS_SECRET_ACCESS_KEY = "YOUR_AWS_SECRET_ACCESS_KEY"
AWS_REGION = "us-west-1"
AWS_DEVICE = "Cepheus-1-108Q"

# Set as process env vars so boto3 / the Braket SDK pick them up automatically.
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

## 2. List available Braket devices

`AWS_DEVICE` above may go stale (providers rename/retire devices). Run this
to confirm the exact current name before submitting any jobs — pick a
Rigetti QPU from the printed list and update `AWS_DEVICE` if needed.

In [ ]:
from qiskit_braket_provider import BraketProvider

# BraketProvider searches all AWS regions for QPUs, so AWS_REGION doesn't
# limit this list; it only excludes devices your account can't access.
for backend in BraketProvider().backends():
    print(f"{backend.name!r:20} {backend.description}")

## 3. Budget for the full catalog

Braket charges a flat **$0.30 per task + $0.000425 per shot**, i.e. **$0.7352
per job** at 1,024 shots, regardless of qubit count or circuit depth.
Submitting every config below once costs:

| Algorithm | Jobs | Configs | Cost (USD) |
|---|---|---|---|
| Grover | 16 | same 16 already run on Ankaa-3 | $11.76 |
| QFT | 18 | same 18 already run on Ankaa-3 | $13.23 |
| BV | 7 | BV2-ALT .. BV8-ALT | $5.15 |
| BVsb | 3 | BVSB27, BVSB28, BVSB29 | $2.21 |
| **Total** | **44** | | **$32.35** |

You don't have to run everything in one sitting — each section below is
independent, so submit whichever algorithm(s) fit your budget right now.

## 4. Reusable submit / retrieve helpers

Every algorithm below follows the same two-step pattern, so define it once
here: `submit_all` fires off every config without waiting, `retrieve_all`
checks AWS for whichever jobs have finished. Both default to the credentials
and device set in Section 1.

In [ ]:
def submit_all(experiment, config_ids, repeats=1, device=None, region=None, key=None, secret=None):
    """Submit every config_id without waiting; returns a list of (config_id, status).

    Set repeats > 1 to submit multiple independent jobs per config (none of
    the sections below need this, but it's here in case you want extra
    statistical repeats for any config).
    """
    device = device or AWS_DEVICE
    region = region or AWS_REGION
    key = key or AWS_ACCESS_KEY_ID
    secret = secret or AWS_SECRET_ACCESS_KEY

    results = []
    for config_id in config_ids:
        for run_idx in range(repeats):
            label = config_id if repeats == 1 else f"{config_id} (run {run_idx + 1}/{repeats})"
            print(f"\n=== {label} ===")
            try:
                result = experiment.run(
                    config_id=config_id,
                    device_id=device,
                    region=region,
                    aws_access_key_id=key,
                    aws_secret_access_key=secret,
                    wait_for_results=False,
                )
                results.append((config_id, result["status"]))
            except Exception as exc:
                # Keep going through the remaining configs even if one submission fails.
                results.append((config_id, f"error: {exc}"))
    return results


def retrieve_all(experiment, device=None, region=None, key=None, secret=None):
    """Check AWS for every pending job of this experiment and print status.

    Safe to re-run repeatedly (immediately, in 10 minutes, tomorrow) — it
    only touches jobs still submitted/queued/running.
    """
    experiment.update_from_aws(
        device_id=device or AWS_DEVICE,
        region=region or AWS_REGION,
        aws_access_key_id=key or AWS_ACCESS_KEY_ID,
        aws_secret_access_key=secret or AWS_SECRET_ACCESS_KEY,
    )
    experiment.print_data_status()

## 5. Grover — submit

Re-runs the same 16 configs already on record for Ankaa-3 (see
`run_aws_experiments.sh`), so the two devices are directly comparable.

In [ ]:
GROVER_CONFIGS = [
    "ASYM-1", "ASYM-2", "M3-2", "SYM-1", "SYM-2", "H3-3", "H3-2", "S3-1", "M3-1",
    "M4-4", "M4-2", "S4-1", "H4-4",
    "S5-1", "S7-1", "S8-1",
]

grover_experiment = GroverAWSExperiment(shots=1024, timeout=600)
submit_all(grover_experiment, GROVER_CONFIGS)

## 6. Grover — retrieve

Re-run as many times as needed while jobs work through the AWS queue.

In [ ]:
retrieve_all(grover_experiment)

## 7. QFT — submit

Re-runs the same 18 configs already on record for Ankaa-3 (see
`run_aws_experiments.sh`), so the two devices are directly comparable.

In [ ]:
QFT_CONFIGS = [
    "SR2", "SR3", "SR4", "SR5", "SR6",
    "PV4-P8", "SP4-P4", "PV4-P4", "PV6-P16", "SP5-P4", "PV6-P8", "SP6-P8",
    "IV4-0000", "IV4-0101",
    "SR8", "SR10", "SP8-P4", "SP10-P4",
]

qft_experiment = QFTAWSExperiment(shots=1024, timeout=600)
submit_all(qft_experiment, QFT_CONFIGS)

## 8. QFT — retrieve

Re-run as many times as needed while jobs work through the AWS queue.

In [ ]:
retrieve_all(qft_experiment)

## 9. BV — submit

Extends the existing n=2-8 alternating-secret ladder used for the
combined DSR plot.

In [ ]:
BV_CONFIGS = [f"BV{n}-ALT" for n in range(2, 9)]  # BV2-ALT .. BV8-ALT

bv_experiment = BVAWSExperiment(shots=1024, timeout=600)
submit_all(bv_experiment, BV_CONFIGS)

## 10. BV — retrieve

Re-run as many times as needed while jobs work through the AWS queue.

In [ ]:
retrieve_all(bv_experiment)

## 11. BVsb — submit

The AWS-compatible (coherent, non-dynamic) variant of the signal-plus-
background campaign — see `bv_signal_background_aws.py` for why this
differs from the IBM (dynamic-circuit) runner.

In [ ]:
bvsb_experiment = BVSignalBackgroundAWSExperiment(shots=1024, timeout=600)
submit_all(bvsb_experiment, bvsb_experiment.get_all_config_ids())

## 12. BVsb — retrieve

Re-run as many times as needed while jobs work through the AWS queue.

In [ ]:
retrieve_all(bvsb_experiment)

## 13. Overall status and next steps

Convenience cell that re-checks every algorithm you submitted above in
one go (skips any you didn't run this session). Once everything shows
`completed`, the JSON files are ready. From a repo checkout, run (in
order) `enrich_dsr_profile.py`, `enrich_hellinger.py`,
`build_csv_from_json.py`, then `differential_success_rate_analysis.py`
to fold them into `DSR_result.csv` and regenerate the figures —
including the new Ankaa-3 vs Cepheus-1-108Q device-comparison plot.

In [ ]:
for exp_name in ("grover_experiment", "qft_experiment", "bv_experiment", "bvsb_experiment"):
    if exp_name in dir():
        retrieve_all(eval(exp_name))
        print()

## 14. Manual recovery (if a retrieve cell finds nothing)

`retrieve_all()` only looks at the local JSON file written at submit time.
If that file is missing — runtime restarted, you re-ran the install cell in
[1] (`--force-reinstall` wipes and recreates the package directory it lives
in), etc. — it will report "nothing to update" even though your job is
still on AWS. As long as you still have the **Job ID** printed by the
submit cell, retrieve it directly here (no local file needed):

In [ ]:
# Swap in the matching experiment class (GroverAWSExperiment, QFTAWSExperiment,
# BVAWSExperiment, or BVSignalBackgroundAWSExperiment) and fill in the Job ID +
# config_id printed when you submitted it. Creating a fresh instance here works
# even if you haven't run that algorithm's "submit" cell in this session.
recovery_experiment = BVAWSExperiment(shots=1024, timeout=600)
recovery_experiment.retrieve_by_job_id(
    job_id="arn:aws:braket:us-west-1:265556963705:quantum-task/8c14c7f1-2dd8-403b-92fb-5c23f7cfc207",
    config_id="BV2-ALT",
    device_id=AWS_DEVICE,
    region=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)